## 1. Install Dependencies


Install the depencies needed to run this notebook and clone the github which contains helper scripts

In [ ]:
!apt-get update && apt-get install -y libsndfile1 ffmpeg
!pip install Cython packaging
!pip install nemo_toolkit['asr'] sentencepiece torchaudio scikit-learn
!git clone https://github.com/diarray-hub/bambara-asr.git # Git repo with the scripts

## 2. Imports and Utilities
Load all necessary modules and helper functions.

In [ ]:
!cp bambara-asr/utils/python/hf_to_nemo_asr.py ./bambara-asr/rlnf/reward/ # Copy the hf_to_nemo_asr.py script to the current directory
# Change directory to the reward subpackage
%cd bambara-asr/rlnf/reward/

In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from rlnf.dataloaders.reward_dataset import get_dataloaders
from rlnf.reward.reward_model import RewardModel
from rlnf.reward.train_utils import fit, evaluate
from sentencepiece import SentencePieceProcessor

In [ ]:
# Load a pre-trained Tokenizer
def load_tokenizer(model_path: str) -> SentencePieceProcessor:
    sp = SentencePieceProcessor()
    sp.Load(model_path)
    return sp

## 3. Configuration

In [ ]:
# Load and save dataset in manifest format
!python hf_to_nemo_asr.py --repo_id=RobotsMali/transcription-scorer --subset=partially-reviewed --save_dir=./data/

In [ ]:
### Paths to manifests and tokenizer
train_manifest = '/path/to/train-manifest.jsonl'
test_manifest = '/path/to/test-manifest.jsonl'
tokenizer_path = '/path/to/tokenizer.model'         # In the GitHub repository

### Hyperparameters
epochs = 5  # Number of epochs to train
batch_size = 16     # Single Batch size for training and testting
lr = 1e-3   # Learning rate used by the optimizer
save_dir = './training_archives'  # Directory to save checkpoints
checkpoint_dir = f'{save_dir}/checkpoints'

use_scheduler = True    # Whether to use a learning rate scheduler (if False, LR will be constant throughout training)
scheduler_step_size = 2  # Number of epochs before reducing LR (if use_scheduler)
scheduler_gamma = 0.8    # Factor by which to reduce LR (if use_scheduler)

### Audio preprocessor Config
preprocessor_config = {
    'normalize': 'per_feature',
    'window_size': 0.02,
    'sample_rate': 16000,
    'window_stride': 0.01,
    'window': 'hann',
    'features': 64,
    'n_fft': 512,
    'frame_splicing': 1,
    'dither': 1e-05,
    'stft_conv': False
}

## Model architecture
# Define the architecture of the reward model
# Basically a regression model with two encoders (audio and text) and a regression head.
embed_dim = 128     # Embedding size of the text encoder
lstm_hidden = 128  # Hidden size of the LSTM encoder used for text
lstm_layers = 1    # Number of Bidirectional LSTM layers used for text

audio_conv_channels = 128  # Number of convolutional channels used for audio
audio_conv_layers = 3   # Number of convolutional layers used for audio
n_mel = preprocessor_config['features']     # Number of melSpectogram input features of the model
dropout = 0.3   # Dropout rate used in the regressor
mlp_hidden_dim = 256     # Hidden size of the MLP regressor (2-Layer Regressor)
seed = 42   # Random seed for reproducibility
plot = True     # Whether to plot the training and validation loss

# Reproducibility
torch.manual_seed(seed)
np.random.seed(seed)

# Create save directory
os.makedirs(save_dir, exist_ok=True)

# Device selection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 4. Load Tokenizer

In [ ]:
print("Loading tokenizer...")
tokenizer = load_tokenizer(tokenizer_path)
vocab_size = tokenizer.GetPieceSize()
print(f"Vocabulary size: {vocab_size}")

## 5. Prepare DataLoaders

In [ ]:
print("Preparing dataloaders...")
# Return two data loaders for training and testing
train_loader, test_loader = get_dataloaders(
    train_manifest,
    test_manifest,
    tokenizer_path,
    preprocessor_config,
    batch_size=batch_size,
    audio_transform=None,
    num_workers=4,   # Note: if yuou are using a GPU, set this to 0, before we fix the issue
)

print(f"Number of training samples: {len(train_loader.dataset)}")
print(f"Number of test samples: {len(test_loader.dataset)}")

## 6. Instantiate Model, Optimizer, Loss, Scheduler

In [ ]:
print("Building model...")
model = RewardModel(
    n_mel=n_mel,
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    lstm_hidden=lstm_hidden,
    lstm_layers=lstm_layers,
    audio_conv_channels=audio_conv_channels,
    audio_conv_layers=audio_conv_layers,
    head_hidden=mlp_hidden_dim,
    dropout=dropout,
)
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.MSELoss()
scheduler = None
if use_scheduler:
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=scheduler_step_size, gamma=scheduler_gamma)
    print(f"Scheduler: StepLR(step_size={scheduler_step_size}, gamma={scheduler_gamma})")

## 7. Training Loop

In [ ]:
# Train the Reward model for {epochs} number of iterations over all training batches
history = fit(
    model=model,
    train_dataloader=train_loader,
    valid_dataloader=test_loader,
    epochs=epochs,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    checkpoint_dir=checkpoint_dir,
    scheduler=scheduler,
)

## 8. Save Final Model & Logs

In [ ]:
final_path = os.path.join(save_dir, 'reward_model.ckpt')
model.save(final_path)

logs_path = os.path.join(save_dir, 'training_logs.json')
with open(logs_path, 'w', encoding='utf-8') as f:
    json.dump(history, f, indent=2)
print(f"Saved training logs to {logs_path}")

## 9. Evaluation & Plots

In [ ]:
print("Running final evaluation...")
evaluate(model, test_loader, criterion, device)

if plot:
    preds, targets = [], []
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            audio = batch['audio_batch'].to(device)
            text = batch['text_batch'].to(device)
            labels = batch['score_batch'].cpu().numpy()
            audio_lens = batch['audio_lengths'].to(device)
            text_lens = batch['text_lengths'].to(device)

            out = model(audio, audio_lens, text, text_lens).cpu().numpy()
            preds.append(out)
            targets.append(labels)
    preds = np.concatenate(preds)
    targets = np.concatenate(targets)

    # Loss curves
    plt.figure()
    plt.plot(range(1, epochs+1), history['train_loss'], label='Train Loss')
    plt.plot(range(1, epochs+1), history['val_loss'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

    # Predictions vs Targets  
    plt.figure()
    plt.scatter(targets, preds, alpha=0.5)
    plt.xlabel('Targets')
    plt.ylabel('Predictions')
    plt.title('Pred vs Target')
    plt.plot([0,1],[0,1], 'r--')
    plt.show()